In [51]:
import os, json, tempfile
from typing import Dict, List, Optional
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

In [52]:
# ========= 1) 在这里配置 =========
VIDEO_PATHS = [
    "dataset/videos/zhangwenze_2.mp4",
    # 想标哪个就把路径填进来；可随时增/删
]
CORPUS_JSON = "dataset/ann/corpus.json"
WIN_SEC = 2.0      # 切片长度(秒)
STEP_SEC = 2.0     # 切片步长(秒)；想重叠可设 < WIN_SEC
TARGET_FPS = 30    # 读取不到fps时的备用

# 中文字体路径（请改成你机器上的字体）
# 例：Windows: C:/Windows/Fonts/msyh.ttc ；macOS: /System/Library/Fonts/PingFang.ttc
#     Linux: /usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc（可能因发行版不同）
# FONT_PATH = "C:\Windows\Fonts\Microsoft YaHei.otf"
FONT_PATH = "C:/Windows/Fonts/msyh.ttc"
FONT_SIZE_SMALL  = 18
FONT_SIZE_MEDIUM = 20
FONT_SIZE_LARGE  = 22

In [53]:
# ================= 键位配置 =================
LABEL_KEYS = {
    ord('1'): "look_left",
    ord('2'): "look_right",
    ord('3'): "look_down",
    ord('4'): "look_offscreen",
    ord('5'): "face_not_visible",
    ord('6'): "talking",
    ord('7'): "other_person_present",
    ord('8'): "other_limb_present",
    ord('9'): "multi_face",
    ord('0'): "leave_seat",
}
REASON_KEYS = {
    ord('g'): "glare",
    ord('b'): "blur",
    ord('f'): "freeze",
    ord('o'): "occlusion",
    ord('a'): "ambiguous",
    ord('u'): "under_exposure",
    ord('v'): "over_exposure",
    ord('m'): "low_bitrate",
    ord('c'): "codec_issue",
    ord('x'): "off_frame",
}
HELP = """键位:
[标签] 1:left 2:right 3:down 4:offscreen 5:face_not 6:talking 7:other_person 8:other_limb 9:multi_face 0:leave_seat
[整段赋值] 数字键切换(可多选) → Space 确认整段
[起止打点] 同一标签键：第一次=开始，第二次=结束（同段可多次）
[ignore] i 开/关；g/b/f/o/a/u/v/m/c/x 设 reason；Enter 自定义 reason
[编辑] c 清空未结束事件和整段选择 | Backspace 撤销最近事件/ignore
[播放] Space 确认并进入下一段 | q 保存并退出
"""

In [54]:
# ========= JSON 读写 =========
def atomic_write_json(path: str, obj: dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with tempfile.NamedTemporaryFile("w", delete=False, dir=os.path.dirname(path),
                                     suffix=".tmp", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        tmp = f.name
    os.replace(tmp, path)

def load_corpus(path: str) -> dict:
    if os.path.exists(path):
        return json.load(open(path, "r", encoding="utf-8"))
    return {"version": "v1", "labels": {}}

def save_corpus(path: str, corpus: dict):
    atomic_write_json(path, corpus)

def video_id_from_path(p: str) -> str:
    return os.path.splitext(os.path.basename(p))[0]

def ensure_video_entry(corpus: dict, video_path: str,
                       win_sec: float, step_sec: float, target_fps: int):
    vid = video_id_from_path(video_path)
    labels = corpus.setdefault("labels", {})
    if vid in labels:
        return vid
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or float(target_fps)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = frames / fps if fps > 0 else 0.0
    cap.release()

    if not step_sec or step_sec <= 0:
        step_sec = win_sec

    segments, t, sid = [], 0.0, 1
    while t < duration - 1e-6:
        st, ed = t, min(t + win_sec, duration)
        segments.append({"id": sid, "start": round(st,3), "end": round(ed,3), "done": False})
        sid += 1; t += step_sec

    labels[vid] = {
        "video_id": vid,
        "video_path": video_path,
        "fps": int(round(fps)) if fps else target_fps,
        "segments": segments,
        "events": [],
        "ignore": [],
        "metadata": {"resolution": {"width": width, "height": height}}
    }

def mark_segment_done(corpus: dict, video_id: str, seg_id: int):
    for s in corpus["labels"][video_id]["segments"]:
        if s["id"] == seg_id:
            s["done"] = True
            return

In [55]:
# ========= 中文绘字 =========
def _load_font(size):
    try:
        return ImageFont.truetype(FONT_PATH, size)
    except Exception:
        return ImageFont.load_default()

def draw_text_cn(img_bgr, lines, xy_list, sizes, colors):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil_img)
    for text, (x, y), size, (b,g,r) in zip(lines, xy_list, sizes, colors):
        font = _load_font(size)
        draw.text((x, y), text, font=font, fill=(r, g, b))
    return cv2.cvtColor(np.asarray(pil_img), cv2.COLOR_RGB2BGR)

In [56]:
# ========= 叠加 UI（含时间轴）=========
"""
def draw_overlay(frame, seg, t_now, fps, chosen_labels, active_events, active_ignore_reason, idx, total):
    h, w = frame.shape[:2]
    panel_h = 180
    cv2.rectangle(frame, (0,0), (w, panel_h), (0,0,0), -1)

    start_t, end_t = seg["start"], seg["end"]
    dur = max(1e-6, end_t - start_t)
    rel = (t_now - start_t) / dur
    rel = 0.0 if rel < 0 else (1.0 if rel > 1 else rel)

    # 时间轴区域
    bar_y1 = panel_h - 30
    bar_y2 = panel_h - 12
    bar_x1 = 10
    bar_x2 = w - 10
    # 背景条
    cv2.rectangle(frame, (bar_x1, bar_y1), (bar_x2, bar_y2), (40,40,40), -1)
    # 进度
    cur_x = int(bar_x1 + rel * (bar_x2 - bar_x1))
    cv2.rectangle(frame, (bar_x1, bar_y1), (cur_x, bar_y2), (60,180,255), -1)
    # 游标
    cv2.line(frame, (cur_x, bar_y1-6), (cur_x, bar_y2+6), (255,255,255), 2)

    # 文本信息
    line1 = f"切片 {idx}/{total}  [{start_t:.2f}-{end_t:.2f}s]  id={seg['id']}"
    line2 = f"t={t_now:.2f}s   dur={dur:.2f}s   fps={int(round(fps))}"
    line3 = "整段选择: " + (", ".join(sorted(chosen_labels)) if chosen_labels else "(无)")
    line4 = "起止事件: " + (", ".join([f"{k}" for k in active_events.keys()]) if active_events else "(无)")
    line5 = f"IGNORE: reason={active_ignore_reason}" if active_ignore_reason else ""
    line6 = "Space=确认/下一段   z=清空   i=Ignore   q=保存退出   鼠标:点击/拖动时间轴  ←/→:±0.10s  Shift+←/→:±0.50s"

    lines = [line1, line2, line3, line4, line5, line6]
    draw_lines, xy, sizes, colors = [], [], [], []
    y = 10
    for i, L in enumerate(lines):
        if not L and i != 4:  # 第5行允许空
            continue
        draw_lines.append(L)
        xy.append((10, y))
        sizes.append(FONT_SIZE_LARGE if i == 0 else FONT_SIZE_MEDIUM)
        colors.append((0,255,255) if i==0 else (200,200,200) if i in (1,5) else (255,255,255))
        y += 26
    frame[:] = draw_text_cn(frame, draw_lines, xy, sizes, colors)

    # 返回时间轴矩形，以便点击检测
    return (bar_x1, bar_y1, bar_x2, bar_y2)
"""

'\ndef draw_overlay(frame, seg, t_now, fps, chosen_labels, active_events, active_ignore_reason, idx, total):\n    h, w = frame.shape[:2]\n    panel_h = 180\n    cv2.rectangle(frame, (0,0), (w, panel_h), (0,0,0), -1)\n\n    start_t, end_t = seg["start"], seg["end"]\n    dur = max(1e-6, end_t - start_t)\n    rel = (t_now - start_t) / dur\n    rel = 0.0 if rel < 0 else (1.0 if rel > 1 else rel)\n\n    # 时间轴区域\n    bar_y1 = panel_h - 30\n    bar_y2 = panel_h - 12\n    bar_x1 = 10\n    bar_x2 = w - 10\n    # 背景条\n    cv2.rectangle(frame, (bar_x1, bar_y1), (bar_x2, bar_y2), (40,40,40), -1)\n    # 进度\n    cur_x = int(bar_x1 + rel * (bar_x2 - bar_x1))\n    cv2.rectangle(frame, (bar_x1, bar_y1), (cur_x, bar_y2), (60,180,255), -1)\n    # 游标\n    cv2.line(frame, (cur_x, bar_y1-6), (cur_x, bar_y2+6), (255,255,255), 2)\n\n    # 文本信息\n    line1 = f"切片 {idx}/{total}  [{start_t:.2f}-{end_t:.2f}s]  id={seg[\'id\']}"\n    line2 = f"t={t_now:.2f}s   dur={dur:.2f}s   fps={int(round(fps))}"\n    line3 = "整

In [57]:
# ========= 叠加 UI（独立面板版本，不覆盖视频）=========
def draw_overlay(frame, seg, t_now, fps,
                 chosen_labels, active_events, active_ignore_reason,
                 idx, total):
    """
    返回: composed_frame, (bar_x1, bar_y1, bar_x2, bar_y2)
    - composed_frame: 上半是原视频，下半是UI面板
    - bar_*: 时间轴在“整张图像坐标系”中的位置（鼠标检测用）
    """

    h, w = frame.shape[:2]
    panel_h = 180

    # 1) 先创建一块独立的UI面板（全黑底）
    panel = np.zeros((panel_h, w, 3), dtype=np.uint8)

    # 2) 在“面板”上画时间轴
    start_t, end_t = seg["start"], seg["end"]
    dur = max(1e-6, end_t - start_t)
    rel = (t_now - start_t) / dur
    rel = 0.0 if rel < 0 else (1.0 if rel > 1 else rel)

    bar_y1 = panel_h - 30
    bar_y2 = panel_h - 12
    bar_x1 = 10
    bar_x2 = w - 10

    # 背景条
    cv2.rectangle(panel, (bar_x1, bar_y1), (bar_x2, bar_y2), (40,40,40), -1)
    # 进度
    cur_x = int(bar_x1 + rel * (bar_x2 - bar_x1))
    cv2.rectangle(panel, (bar_x1, bar_y1), (cur_x, bar_y2), (60,180,255), -1)
    # 游标
    cv2.line(panel, (cur_x, bar_y1-6), (cur_x, bar_y2+6), (255,255,255), 2)

    # 3) 在“面板”上画文字（Pillow中文）
    line1 = f"切片 {idx}/{total}  [{start_t:.2f}-{end_t:.2f}s]  id={seg['id']}"
    line2 = f"t={t_now:.2f}s   dur={dur:.2f}s   fps={int(round(fps))}"
    line3 = "整段选择: " + (", ".join(sorted(chosen_labels)) if chosen_labels else "(无)")
    line4 = "起止事件: " + (", ".join([f"{k}" for k in active_events.keys()]) if active_events else "(无)")
    line5 = f"IGNORE: reason={active_ignore_reason}" if active_ignore_reason else ""
    line6 = "Space=确认/下一段   c=清空   i=Ignore   q=保存退出   鼠标:点击/拖动时间轴  ←/→:±0.10s"

    lines = [line1, line2, line3, line4, line5, line6]
    draw_lines, xy, sizes, colors = [], [], [], []
    y = 10
    for i, L in enumerate(lines):
        if not L and i != 4:  # 第5行允许空
            continue
        draw_lines.append(L)
        xy.append((10, y))
        sizes.append(FONT_SIZE_LARGE if i == 0 else FONT_SIZE_MEDIUM)
        colors.append((0,255,255) if i==0 else (200,200,200) if i in (1,5) else (255,255,255))
        y += 26
    # 用你已有的 draw_text_cn 在“panel”上绘字
    panel[:] = draw_text_cn(panel, draw_lines, xy, sizes, colors)

    # 4) 把“视频 + 面板”上下拼接
    composed = np.vstack([frame, panel])

    # 5) 返回时间轴的“全图坐标”给鼠标命中检测（y 轴别忘了 +h）
    return composed, (bar_x1, h + bar_y1, bar_x2, h + bar_y2)


In [58]:
# ========= 标注一个视频 =========
def label_one_video(corpus_path: str, video_path: str,
                    win_sec: float, step_sec: float, target_fps: int):
    corpus = load_corpus(corpus_path)
    ensure_video_entry(corpus, video_path, win_sec, step_sec, target_fps)
    save_corpus(corpus_path, corpus)

    vid = video_id_from_path(video_path)
    entry = corpus["labels"][vid]
    segs = entry["segments"]
    total = len(segs)
    if total == 0:
        print(f"[WARN] {video_path} 无切片"); return

    # 找到第一个未完成的切片
    start_idx = next((i for i,s in enumerate(segs) if not s.get("done", False)), None)
    if start_idx is None:
        print(f"[OK] {video_path} 全部切片已完成"); return

    cap = cv2.VideoCapture(video_path)
    real_fps = cap.get(cv2.CAP_PROP_FPS)
    fps = real_fps if real_fps and real_fps > 1e-3 else entry.get("fps", target_fps)

    print("\n" + HELP)
    win_name = "Labeler (CN+Timeline)"
    cv2.namedWindow(win_name)

    # 鼠标交互状态
    mouse = {"dragging": False, "bar": (0,0,0,0), "curr_ptr": 0}

    def set_curr_by_ratio(seg, ratio):
        ratio = max(0.0, min(1.0, ratio))
        t = seg["start"] + ratio * (seg["end"] - seg["start"])
        return t

    def on_mouse(event, x, y, flags, param):
        seg, fps_local = param["seg"], param["fps"]
        bar_x1, bar_y1, bar_x2, bar_y2 = mouse["bar"]
        if event == cv2.EVENT_LBUTTONDOWN:
            if bar_x1 <= x <= bar_x2 and bar_y1-8 <= y <= bar_y2+8:
                mouse["dragging"] = True
                r = (x - bar_x1) / max(1, (bar_x2 - bar_x1))
                t = set_curr_by_ratio(seg, r)
                param["curr_frame"][0] = int(t * fps_local)
        elif event == cv2.EVENT_MOUSEMOVE and mouse["dragging"]:
            r = (x - bar_x1) / max(1, (bar_x2 - bar_x1))
            t = set_curr_by_ratio(seg, r)
            param["curr_frame"][0] = int(t * fps_local)
        elif event == cv2.EVENT_LBUTTONUP:
            mouse["dragging"] = False

    for idx in range(start_idx, total):
        seg = segs[idx]
        if seg.get("done", False):
            continue

        start_t, end_t = seg["start"], seg["end"]
        start_f = int(start_t * fps)
        end_f = max(int(end_t * fps) - 1, start_f)
        curr = start_f

        buffer_events: List[Dict] = []
        buffer_ignores: List[Dict] = []

        chosen_labels = set()
        active_events: Dict[str, float] = {}
        ignore_active = False
        ignore_start = None
        active_ignore_reason = None

        # 注册鼠标回调（传可变容器保存 curr）
        curr_holder = [curr]
        cv2.setMouseCallback(win_name, on_mouse, param={"seg": seg, "fps": fps, "curr_frame": curr_holder})

        confirmed = False
        while not confirmed:
            curr = curr_holder[0]
            # 边界夹取
            if curr < start_f: curr = start_f
            if curr > end_f:   curr = end_f
            curr_holder[0] = curr

            cap.set(cv2.CAP_PROP_POS_FRAMES, curr)
            ret, frame = cap.read()
            if not ret:
                curr = start_f
                curr_holder[0] = curr
                continue

            t_now = curr / fps
            # mouse["bar"] = draw_overlay(frame, seg, t_now, fps,
            #                             chosen_labels, active_events, active_ignore_reason,
            #                             idx+1, total)

            frame, mouse["bar"] = draw_overlay(frame, seg, t_now, fps,
                                   chosen_labels, active_events, active_ignore_reason,
                                   idx+1, total)
            cv2.imshow(win_name, frame)
            key = cv2.waitKey(int(1000//fps)) & 0xFF

            # 方向键微调
            if key == 81:  # Left
                step = 0.5 if (cv2.getWindowProperty(win_name, 0) and (cv2.waitKey(1) & 0xFF) == 0) else 0.1
                # 上面方式获取修饰键不稳定，直接用较简单的：Left=0.1s；加Shift可改成另外键，自行扩展
                curr = max(start_f, curr - int(0.1 * fps))
                curr_holder[0] = curr
                continue
            elif key == 83:  # Right
                curr = min(end_f, curr + int(0.1 * fps))
                curr_holder[0] = curr
                continue

            if key in LABEL_KEYS:
                label = LABEL_KEYS[key]
                if label in active_events:
                    ev_start = max(start_t, active_events[label]); ev_end = min(t_now, end_t)
                    if ev_end > ev_start:
                        buffer_events.append({"label": label, "start": round(ev_start,3), "end": round(ev_end,3)})
                    del active_events[label]
                else:
                    active_events[label] = t_now
                # 同步整段选择切换
                if label in chosen_labels: chosen_labels.remove(label)
                else: chosen_labels.add(label)

            elif key == ord('i'):
                if not ignore_active:
                    ignore_active = True; ignore_start = t_now
                    active_ignore_reason = active_ignore_reason or "manual"
                else:
                    ig_start = max(start_t, ignore_start); ig_end = min(t_now, end_t)
                    if ig_end > ig_start:
                        buffer_ignores.append({"start": round(ig_start,3), "end": round(ig_end,3),
                                               "reason": active_ignore_reason or "manual"})
                    ignore_active = False; ignore_start = None; active_ignore_reason = None

            elif key in REASON_KEYS:
                active_ignore_reason = REASON_KEYS[key]

            elif key == 13:  # Enter：自定义 reason
                try:
                    r = input("输入自定义 ignore reason: ").strip()
                except EOFError:
                    r = ""
                if r:
                    active_ignore_reason = r

            elif key == ord('z'):
                chosen_labels.clear(); active_events.clear()

            elif key == 8:  # Backspace 撤销最近
                if buffer_events:
                    buffer_events.pop(-1)
                elif buffer_ignores:
                    buffer_ignores.pop(-1)

            elif key == ord(' '):  # 确认本段
                if active_events:
                    for lb, st in list(active_events.items()):
                        ev_start = max(start_t, st); ev_end = end_t
                        if ev_end > ev_start:
                            buffer_events.append({"label": lb, "start": round(ev_start,3), "end": round(ev_end,3)})
                    active_events.clear()
                if ignore_active:
                    ig_start = max(start_t, ignore_start); ig_end = end_t
                    if ig_end > ig_start:
                        buffer_ignores.append({"start": round(ig_start,3), "end": round(ig_end,3),
                                               "reason": active_ignore_reason or "manual"})
                    ignore_active = False; active_ignore_reason = None
                for lb in sorted(list(chosen_labels)):
                    buffer_events.append({"label": lb, "start": round(start_t,3), "end": round(end_t,3)})
                chosen_labels.clear()

                # 写回
                corpus = load_corpus(CORPUS_JSON)
                entry = corpus["labels"][vid]
                entry.setdefault("events", []).extend(buffer_events)
                entry.setdefault("ignore", []).extend(buffer_ignores)
                mark_segment_done(corpus, vid, seg_id=seg["id"])
                save_corpus(CORPUS_JSON, corpus)
                buffer_events.clear(); buffer_ignores.clear()
                confirmed = True

            elif key == ord('q'):
                if buffer_events or buffer_ignores:
                    corpus = load_corpus(CORPUS_JSON)
                    entry = corpus["labels"][vid]
                    entry.setdefault("events", []).extend(buffer_events)
                    entry.setdefault("ignore", []).extend(buffer_ignores)
                    save_corpus(CORPUS_JSON, corpus)
                cap.release(); cv2.destroyAllWindows()
                return

            # 自动循环播放（如果没有拖动/按键改变 curr）
            if not mouse["dragging"]:
                curr += 1
                if curr > end_f:
                    curr = start_f
                curr_holder[0] = curr

    cap.release(); cv2.destroyAllWindows()

In [59]:
# ========= 主流程 =========
def main():
    if not VIDEO_PATHS:
        print("[ERR] 请在 VIDEO_PATHS 中填入要标注的视频路径"); return
    for i, vp in enumerate(VIDEO_PATHS, 1):
        if not os.path.exists(vp):
            print(f"[WARN] 找不到视频：{vp}，跳过"); continue
        print(f"\n========== [{i}/{len(VIDEO_PATHS)}] {os.path.basename(vp)} ==========")
        label_one_video(CORPUS_JSON, vp, WIN_SEC, STEP_SEC, TARGET_FPS)
        print(f"[OK] 已写回 {CORPUS_JSON}")

if __name__ == "__main__":
    main()


========== [1/1] zhangwenze_2.mp4 ==========

键位:
[标签] 1:left 2:right 3:down 4:offscreen 5:face_not 6:talking 7:other_person 8:other_limb 9:multi_face 0:leave_seat
[整段赋值] 数字键切换(可多选) → Space 确认整段
[起止打点] 同一标签键：第一次=开始，第二次=结束（同段可多次）
[ignore] i 开/关；g/b/f/o/a/u/v/m/c/x 设 reason；Enter 自定义 reason
[编辑] c 清空未结束事件和整段选择 | Backspace 撤销最近事件/ignore
[播放] Space 确认并进入下一段 | q 保存并退出

[OK] 已写回 dataset/ann/corpus.json
